In [2]:
import os
import glob
import torch
import torch.nn.functional as F
import numpy as np
import time
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from torch_geometric.datasets import Planetoid, WikiCS, Amazon
from torch_geometric.transforms import NormalizeFeatures
from layers import Classifier
from models.GraFN import GraFN
from torch_geometric.nn import GCNConv, GATConv, SAGEConv
import torch.nn as nn

# -------------------------------
# Encoder definitions
# -------------------------------
class GCNEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, out_dim)
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x

class GATEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, heads=8):
        super().__init__()
        self.conv1 = GATConv(in_dim, hidden_dim, heads=heads, concat=True)
        self.conv2 = GATConv(hidden_dim * heads, out_dim, heads=1, concat=False)
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.elu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x

class GraphSAGEEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, out_dim)
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x

# -------------------------------
# Datasets
# -------------------------------
datasets_info = {
    "Cora": lambda: Planetoid(root='data/Planetoid', name='Cora', transform=NormalizeFeatures()),
    "CiteSeer": lambda: Planetoid(root='data/Planetoid', name='CiteSeer', transform=NormalizeFeatures()),
    "PubMed": lambda: Planetoid(root='data/Planetoid', name='PubMed', transform=NormalizeFeatures()),
    "WikiCS": lambda: WikiCS(root='data/WikiCS', transform=NormalizeFeatures()),
    "AmazonPhoto": lambda: Amazon(root='data/Amazon', name='Photo', transform=NormalizeFeatures())
}

mask_dirs_template = "/Users/sujan/Modularity based semi supervised learning/masks/{dataset}/{split}"
splits = ["30_70", "70_30"]

base_results_dir = "/Users/sujan/Downloads/GraFN/results"
base_embeddings_dir = "/Users/sujan/Downloads/GraFN/embeddings"

# -------------------------------
# Pipeline per mask and encoder
# -------------------------------
def run_pipeline(data, mask_file, dataset_name, split_type, encoder_class, encoder_name, embedding_dim=150, epochs=200):
    num_nodes = data.num_nodes
    test_indices = np.load(mask_file)
    test_mask = torch.zeros(num_nodes, dtype=torch.bool)
    test_mask[test_indices] = True
    train_mask = ~test_mask
    data.train_mask = train_mask
    data.test_mask = test_mask

    print(f"\n=== Processing Mask File: {os.path.basename(mask_file)} ({split_type}, {encoder_name}) ===")
    print(f"Train nodes: {train_mask.sum().item()} | Test nodes: {test_mask.sum().item()}")

    # Model
    in_features = embedding_dim
    hidden_dim = 128
    out_dim = embedding_dim
    encoder = encoder_class(in_features, hidden_dim, out_dim)
    classifier = Classifier(out_dim, data.y.max().item() + 1)
    model = GraFN(encoder, classifier, unique_labels=list(range(data.y.max().item() + 1)))
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

    # Training
    model.train()
    start_time = time.time()
    for epoch in range(epochs):
        optimizer.zero_grad()
        out = model(data)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()
        if (epoch+1) % 20 == 0 or epoch == 0:
            val_loss = F.cross_entropy(out[data.test_mask], data.y[data.test_mask])
            print(f"Epoch {epoch+1:03d} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f}")
    elapsed_time = time.time() - start_time

    # Save embeddings
    model.eval()
    with torch.no_grad():
        embeddings = model(data)
    mask_basename = os.path.basename(mask_file)
    seed_str = "seed42"
    if "seed" in mask_basename:
        parts = mask_basename.split("seed")
        if len(parts) > 1:
            seed_str = "seed" + parts[1].replace(".npy", "")
    embeddings_subdir = os.path.join(base_embeddings_dir, dataset_name, split_type)
    os.makedirs(embeddings_subdir, exist_ok=True)
    embedding_filename = f"{dataset_name}_{encoder_name}_{split_type}_{seed_str}.pt"
    torch.save(embeddings, os.path.join(embeddings_subdir, embedding_filename))
    print(f"Saved embeddings to {os.path.join(embeddings_subdir, embedding_filename)}")

    # Evaluation
    with torch.no_grad():
        out = model(data)
        preds = out.argmax(dim=1).cpu().numpy()
        true_labels = data.y.cpu().numpy()
        test_mask_np = data.test_mask.cpu().numpy()
        test_preds = preds[test_mask_np]
        test_labels = true_labels[test_mask_np]

        acc = accuracy_score(test_labels, test_preds)
        f1_macro = f1_score(test_labels, test_preds, average="macro")
        f1_micro = f1_score(test_labels, test_preds, average="micro")

    return {
        "encoder": encoder_name,
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "time": elapsed_time
    }

# -------------------------------
# Run experiments
# -------------------------------
encoder_classes = {
    "GCN": GCNEncoder,
    "GAT": GATEncoder,
    "GraphSAGE": GraphSAGEEncoder
}

embedding_dim = 150
epochs = 200

for dataset_name, loader in datasets_info.items():
    print(f"\n========== Dataset: {dataset_name} ==========")
    dataset = loader()
    data = dataset[0]
    data.x = torch.randn(data.num_nodes, embedding_dim)

    for split_type in splits:
        mask_dir = mask_dirs_template.format(dataset=dataset_name, split=split_type)
        results_subdir = os.path.join(base_results_dir, dataset_name, split_type)
        os.makedirs(results_subdir, exist_ok=True)

        mask_files = sorted(glob.glob(os.path.join(mask_dir, "*.npy")))
        if len(mask_files) == 0:
            print(f"No mask files found for {dataset_name} split {split_type}. Skipping.")
            continue

        all_encoder_results = []

        for encoder_name, encoder_class in encoder_classes.items():
            encoder_results = []
            for mask_file in mask_files:
                result = run_pipeline(data, mask_file, dataset_name, split_type, encoder_class, encoder_name, embedding_dim, epochs)
                encoder_results.append(result)

            # Compute mean ± std
            acc_vals = [r["accuracy"] for r in encoder_results]
            f1_macro_vals = [r["f1_macro"] for r in encoder_results]
            f1_micro_vals = [r["f1_micro"] for r in encoder_results]
            time_vals = [r["time"] for r in encoder_results]

            avg_metrics = {
                "encoder": encoder_name,
                "accuracy": f"{np.mean(acc_vals):.4f} ± {np.std(acc_vals):.4f}",
                "f1_macro": f"{np.mean(f1_macro_vals):.4f} ± {np.std(f1_macro_vals):.4f}",
                "f1_micro": f"{np.mean(f1_micro_vals):.4f} ± {np.std(f1_micro_vals):.4f}",
                "time": f"{np.mean(time_vals):.2f} ± {np.std(time_vals):.2f}"
            }
            all_encoder_results.append(avg_metrics)

        # Save averaged results
        results_df = pd.DataFrame(all_encoder_results)
        metrics_df = results_df[['encoder', 'accuracy', 'f1_macro', 'f1_micro']]
        times_df = results_df[['encoder', 'time']]
        metrics_df.to_csv(os.path.join(results_subdir, f"{dataset_name}_results.csv"), index=False)
        times_df.to_csv(os.path.join(results_subdir, f"{dataset_name}_execution_times.csv"), index=False)

        print(f"Saved averaged results and execution times for {dataset_name} split {split_type}")



========== Dataset: Cora ==========

=== Processing Mask File: Cora_30_70_masked_indices_seed123.npy (30_70, GCN) ===
Train nodes: 813 | Test nodes: 1895
Epoch 001 | Train Loss: 5.0445 | Val Loss: 5.0237
Epoch 020 | Train Loss: 0.5364 | Val Loss: 1.2896
Epoch 040 | Train Loss: 0.1225 | Val Loss: 1.5384
Epoch 060 | Train Loss: 0.0514 | Val Loss: 1.6711
Epoch 080 | Train Loss: 0.0428 | Val Loss: 1.6668
Epoch 100 | Train Loss: 0.0424 | Val Loss: 1.6359
Epoch 120 | Train Loss: 0.0399 | Val Loss: 1.6345
Epoch 140 | Train Loss: 0.0358 | Val Loss: 1.6570
Epoch 160 | Train Loss: 0.0326 | Val Loss: 1.6791
Epoch 180 | Train Loss: 0.0304 | Val Loss: 1.6968
Epoch 200 | Train Loss: 0.0286 | Val Loss: 1.7126
Saved embeddings to /Users/sujan/Downloads/GraFN/embeddings/Cora/30_70/Cora_GCN_30_70_seed123.pt

=== Processing Mask File: Cora_30_70_masked_indices_seed2025.npy (30_70, GCN) ===
Train nodes: 813 | Test nodes: 1895
Epoch 001 | Train Loss: 5.0173 | Val Loss: 5.0129
Epoch 020 | Train Loss: 0.526

/Users/sujan/Downloads/GraFN/grafn-env/lib/python3.12/site-packages/torch_geometric/datasets/wikics.py:45: UserWarning: The WikiCS dataset now returns an undirected graph by default. Please explicitly specify 'is_undirected=False' to restore the old behavior.
  warnings.warn(


Epoch 001 | Train Loss: 4.9903 | Val Loss: 4.9906
Epoch 020 | Train Loss: 1.5915 | Val Loss: 1.8498
Epoch 040 | Train Loss: 1.0328 | Val Loss: 1.7229
Epoch 060 | Train Loss: 0.8017 | Val Loss: 1.7964
Epoch 080 | Train Loss: 0.6784 | Val Loss: 1.8451
Epoch 100 | Train Loss: 0.5904 | Val Loss: 1.8812
Epoch 120 | Train Loss: 0.5307 | Val Loss: 1.9048
Epoch 140 | Train Loss: 0.4886 | Val Loss: 1.9314
Epoch 160 | Train Loss: 0.4563 | Val Loss: 1.9512
Epoch 180 | Train Loss: 0.4307 | Val Loss: 1.9637
Epoch 200 | Train Loss: 0.4101 | Val Loss: 1.9746
Saved embeddings to /Users/sujan/Downloads/GraFN/embeddings/WikiCS/30_70/WikiCS_GCN_30_70_seed123.pt

=== Processing Mask File: WikiCS_30_70_masked_indices_seed2025.npy (30_70, GCN) ===
Train nodes: 3511 | Test nodes: 8190
Epoch 001 | Train Loss: 5.0716 | Val Loss: 5.0648
Epoch 020 | Train Loss: 1.5968 | Val Loss: 1.8923
Epoch 040 | Train Loss: 1.0404 | Val Loss: 1.7678
Epoch 060 | Train Loss: 0.8286 | Val Loss: 1.8199
Epoch 080 | Train Loss: 0.7

In [4]:
import os
import glob
import torch
import torch.nn.functional as F
import numpy as np
import time
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from torch_geometric.datasets import Planetoid, WikiCS, Amazon
from torch_geometric.transforms import NormalizeFeatures
from layers import Classifier
from models.GraFN import GraFN
from torch_geometric.nn import GCNConv, GATConv, SAGEConv
import torch.nn as nn

# -------------------------------
# Encoder definitions
# -------------------------------
class GCNEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, out_dim)
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x

class GATEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, heads=8):
        super().__init__()
        self.conv1 = GATConv(in_dim, hidden_dim, heads=heads, concat=True)
        self.conv2 = GATConv(hidden_dim * heads, out_dim, heads=1, concat=False)
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.elu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x

class GraphSAGEEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, out_dim)
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x

# -------------------------------
# Datasets
# -------------------------------
datasets_info = {
    "AmazonPhotos": lambda: Amazon(root='data/Amazon', name='Photo', transform=NormalizeFeatures())
}

mask_dirs_template = "/Users/sujan/Modularity based semi supervised learning/masks/{dataset}/{split}"
splits = ["30_70", "70_30"]

base_results_dir = "/Users/sujan/Downloads/GraFN/results"
base_embeddings_dir = "/Users/sujan/Downloads/GraFN/embeddings"

# -------------------------------
# Pipeline per mask and encoder
# -------------------------------
def run_pipeline(data, mask_file, dataset_name, split_type, encoder_class, encoder_name, embedding_dim=150, epochs=200):
    num_nodes = data.num_nodes
    test_indices = np.load(mask_file)
    test_mask = torch.zeros(num_nodes, dtype=torch.bool)
    test_mask[test_indices] = True
    train_mask = ~test_mask
    data.train_mask = train_mask
    data.test_mask = test_mask

    print(f"\n=== Processing Mask File: {os.path.basename(mask_file)} ({split_type}, {encoder_name}) ===")
    print(f"Train nodes: {train_mask.sum().item()} | Test nodes: {test_mask.sum().item()}")

    # Model
    in_features = embedding_dim
    hidden_dim = 128
    out_dim = embedding_dim
    encoder = encoder_class(in_features, hidden_dim, out_dim)
    classifier = Classifier(out_dim, data.y.max().item() + 1)
    model = GraFN(encoder, classifier, unique_labels=list(range(data.y.max().item() + 1)))
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

    # Training
    model.train()
    start_time = time.time()
    for epoch in range(epochs):
        optimizer.zero_grad()
        out = model(data)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()
        if (epoch+1) % 20 == 0 or epoch == 0:
            val_loss = F.cross_entropy(out[data.test_mask], data.y[data.test_mask])
            print(f"Epoch {epoch+1:03d} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f}")
    elapsed_time = time.time() - start_time

    # Save embeddings
    model.eval()
    with torch.no_grad():
        embeddings = model(data)
    mask_basename = os.path.basename(mask_file)
    seed_str = "seed42"
    if "seed" in mask_basename:
        parts = mask_basename.split("seed")
        if len(parts) > 1:
            seed_str = "seed" + parts[1].replace(".npy", "")
    embeddings_subdir = os.path.join(base_embeddings_dir, dataset_name, split_type)
    os.makedirs(embeddings_subdir, exist_ok=True)
    embedding_filename = f"{dataset_name}_{encoder_name}_{split_type}_{seed_str}.pt"
    torch.save(embeddings, os.path.join(embeddings_subdir, embedding_filename))
    print(f"Saved embeddings to {os.path.join(embeddings_subdir, embedding_filename)}")

    # Evaluation
    with torch.no_grad():
        out = model(data)
        preds = out.argmax(dim=1).cpu().numpy()
        true_labels = data.y.cpu().numpy()
        test_mask_np = data.test_mask.cpu().numpy()
        test_preds = preds[test_mask_np]
        test_labels = true_labels[test_mask_np]

        acc = accuracy_score(test_labels, test_preds)
        f1_macro = f1_score(test_labels, test_preds, average="macro")
        f1_micro = f1_score(test_labels, test_preds, average="micro")

    return {
        "encoder": encoder_name,
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "time": elapsed_time
    }

# -------------------------------
# Run experiments
# -------------------------------
encoder_classes = {
    "GCN": GCNEncoder,
    "GAT": GATEncoder,
    "GraphSAGE": GraphSAGEEncoder
}

embedding_dim = 150
epochs = 200

for dataset_name, loader in datasets_info.items():
    print(f"\n========== Dataset: {dataset_name} ==========")
    dataset = loader()
    data = dataset[0]
    data.x = torch.randn(data.num_nodes, embedding_dim)

    for split_type in splits:
        mask_dir = mask_dirs_template.format(dataset=dataset_name, split=split_type)
        results_subdir = os.path.join(base_results_dir, dataset_name, split_type)
        os.makedirs(results_subdir, exist_ok=True)

        mask_files = sorted(glob.glob(os.path.join(mask_dir, "*.npy")))
        if len(mask_files) == 0:
            print(f"No mask files found for {dataset_name} split {split_type}. Skipping.")
            continue

        all_encoder_results = []

        for encoder_name, encoder_class in encoder_classes.items():
            encoder_results = []
            for mask_file in mask_files:
                result = run_pipeline(data, mask_file, dataset_name, split_type, encoder_class, encoder_name, embedding_dim, epochs)
                encoder_results.append(result)

            # Compute mean ± std
            acc_vals = [r["accuracy"] for r in encoder_results]
            f1_macro_vals = [r["f1_macro"] for r in encoder_results]
            f1_micro_vals = [r["f1_micro"] for r in encoder_results]
            time_vals = [r["time"] for r in encoder_results]

            avg_metrics = {
                "encoder": encoder_name,
                "accuracy": f"{np.mean(acc_vals):.4f} ± {np.std(acc_vals):.4f}",
                "f1_macro": f"{np.mean(f1_macro_vals):.4f} ± {np.std(f1_macro_vals):.4f}",
                "f1_micro": f"{np.mean(f1_micro_vals):.4f} ± {np.std(f1_micro_vals):.4f}",
                "time": f"{np.mean(time_vals):.2f} ± {np.std(time_vals):.2f}"
            }
            all_encoder_results.append(avg_metrics)

        # Save averaged results
        results_df = pd.DataFrame(all_encoder_results)
        metrics_df = results_df[['encoder', 'accuracy', 'f1_macro', 'f1_micro']]
        times_df = results_df[['encoder', 'time']]
        metrics_df.to_csv(os.path.join(results_subdir, f"{dataset_name}_results.csv"), index=False)
        times_df.to_csv(os.path.join(results_subdir, f"{dataset_name}_execution_times.csv"), index=False)

        print(f"Saved averaged results and execution times for {dataset_name} split {split_type}")



========== Dataset: AmazonPhotos ==========

=== Processing Mask File: AmazonPhotos_30_70_masked_indices_seed123.npy (30_70, GCN) ===
Train nodes: 2295 | Test nodes: 5355
Epoch 001 | Train Loss: 4.9809 | Val Loss: 4.9800
Epoch 020 | Train Loss: 0.9502 | Val Loss: 1.1252
Epoch 040 | Train Loss: 0.4202 | Val Loss: 0.8891
Epoch 060 | Train Loss: 0.2843 | Val Loss: 0.8885
Epoch 080 | Train Loss: 0.2391 | Val Loss: 0.9117
Epoch 100 | Train Loss: 0.2066 | Val Loss: 0.9340
Epoch 120 | Train Loss: 0.1816 | Val Loss: 0.9513
Epoch 140 | Train Loss: 0.1651 | Val Loss: 0.9681
Epoch 160 | Train Loss: 0.1527 | Val Loss: 0.9813
Epoch 180 | Train Loss: 0.1428 | Val Loss: 0.9921
Epoch 200 | Train Loss: 0.1351 | Val Loss: 1.0017
Saved embeddings to /Users/sujan/Downloads/GraFN/embeddings/AmazonPhotos/30_70/AmazonPhotos_GCN_30_70_seed123.pt

=== Processing Mask File: AmazonPhotos_30_70_masked_indices_seed2025.npy (30_70, GCN) ===
Train nodes: 2295 | Test nodes: 5355
Epoch 001 | Train Loss: 4.9794 | Val 